# Gemma 4 12B — SRD-4 → .axm → GGUF + Axiom MET Hydration

**What this notebook produces**

| Output file | Size (est.) | Description |
|---|---|---|
| `gemma4_12b_srd4.axm` | ~6.5 GB | Signed SRD-4 container (~4.5 bpw) |
| `gemma4_12b_q4km.gguf` | ~7.5 GB | GGUF Q4_K_M — runs in llama.cpp / PocketPal |
| `gemma4_12b_q4km.axiom_meta.json` | ~5 KB | MET slot map + hydration policy sidecar |

**Hardware required**

| Tier | VRAM | RAM | Works? |
|---|---|---|---|
| **A100 40 GB** | 40 GB | 83 GB | ✓ Recommended |
| A100 80 GB | 80 GB | 83 GB | ✓ Easy |
| T4 High-RAM | 15 GB | 52 GB | ⚠ Tight — device_map=auto |
| T4 Standard | 15 GB | 13 GB | ✗ OOM |

> **Gemma gated model** — you must accept terms at  
> https://huggingface.co/google/gemma-4-12b-it  
> before Cell 2 will download the model.

**Cells**
1. GPU check · clone axiom · clone + build llama.cpp (CUDA)
2. HuggingFace login · download Gemma 4 12B
3. SRD-4 pack → `.axm` (20–40 min)
4. Verify `.axm` proof ledger
5. Extract `.axm` → GGUF Q4_K_M (~15 min)
6. Axiom MET metadata sidecar (embedding slot + transformer chunks)
7. Memory dashboard + smoke test

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — GPU check · clone axiom · clone + cmake-build llama.cpp  (~15 min)
# ══════════════════════════════════════════════════════════════════════════════
import os, secrets, subprocess, sys, time
from pathlib import Path

REPO        = Path('/content/axiom')
REPO_URL    = 'https://github.com/orivael-dev/axiom.git'
REPO_BRANCH = 'claude/srd-multimodal'
LLAMA_DIR   = Path('/content/llama.cpp')
OUT_DIR     = Path('/content')
KEY_FILE    = OUT_DIR / 'axiom_master.key'

# ── GPU info ─────────────────────────────────────────────────────────────────
import torch
p = torch.cuda.get_device_properties(0)
vram_gb = p.total_memory / 1024**3
cuda_arch = p.major * 10 + p.minor        # 80 = A100, 90 = H100, 75 = T4
print(f'GPU  : {p.name}  {vram_gb:.1f} GB VRAM  SM {p.major}.{p.minor}  arch={cuda_arch}')
try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1024**3
    print(f'RAM  : {ram_gb:.1f} GB system')
except ImportError:
    ram_gb = 0

if vram_gb < 30:
    print('\n  ⚠  < 30 GB VRAM — Gemma 4 12B (~24 GB FP16) will spill to system RAM.')
    print('     Recommend: Runtime → Change runtime type → A100')
else:
    print('  ✓ A100 — model fits entirely in VRAM')

# ── Clone axiom ──────────────────────────────────────────────────────────────
if not REPO.is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH,
                    REPO_URL, str(REPO)], check=True)
    print(f'✓ axiom cloned  ({REPO_BRANCH})')
else:
    r = subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'],
                       capture_output=True, text=True)
    print(f'✓ axiom updated  {r.stdout.strip()}')

sys.path.insert(0, str(REPO))

# ── Persist AXIOM_MASTER_KEY ─────────────────────────────────────────────────
if KEY_FILE.is_file() and not os.environ.get('AXIOM_MASTER_KEY'):
    os.environ['AXIOM_MASTER_KEY'] = KEY_FILE.read_text().strip()
    print('AXIOM_MASTER_KEY restored from session key file')
elif not os.environ.get('AXIOM_MASTER_KEY'):
    key = secrets.token_hex(32)
    os.environ['AXIOM_MASTER_KEY'] = key
    KEY_FILE.write_text(key)
    print('AXIOM_MASTER_KEY generated and saved')
else:
    print('AXIOM_MASTER_KEY already set')

# ── pip install ──────────────────────────────────────────────────────────────
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'accelerate', 'psutil',
                'huggingface_hub', 'sentencepiece'], check=True)
print('✓ pip packages installed')

# ── Clone + build llama.cpp ───────────────────────────────────────────────────
# Always pull if the dir exists — a stale cached build is the #1 cause of
# garbage PPL numbers (same issue seen with Gemma 3 before llama.cpp gained
# full support for that architecture).  Pull is a no-op when already current.
_q_bin   = LLAMA_DIR / 'build/bin/llama-quantize'
_cli_bin = LLAMA_DIR / 'build/bin/llama-cli'
_needs_build = False

if not LLAMA_DIR.is_dir():
    print('\nCloning llama.cpp ...')
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/ggerganov/llama.cpp',
                    str(LLAMA_DIR)], check=True)
    _needs_build = True
else:
    r_pull = subprocess.run(
        ['git', '-C', str(LLAMA_DIR), 'pull', '--ff-only'],
        capture_output=True, text=True)
    _already_current = 'Already up to date' in r_pull.stdout
    if _already_current and _q_bin.is_file():
        r_log = subprocess.run(
            ['git', '-C', str(LLAMA_DIR), 'log', '--oneline', '-1'],
            capture_output=True, text=True)
        print(f'✓ llama.cpp up to date  ({r_log.stdout.strip()})')
    else:
        if not _already_current:
            print(f'  llama.cpp updated: {r_pull.stdout.strip()}')
        _needs_build = True

if _needs_build:
    print(f'\nBuilding llama.cpp (CUDA arch={cuda_arch}) — ~10 min ...')
    t0 = time.time()
    subprocess.run(
        ['cmake', '-B', 'build',
         '-DCMAKE_BUILD_TYPE=Release',
         '-DGGML_CUDA=ON',
         f'-DCMAKE_CUDA_ARCHITECTURES={cuda_arch}'],
        cwd=LLAMA_DIR, check=True,
    )
    subprocess.run(
        ['cmake', '--build', 'build',
         '--target', 'llama-quantize', 'llama-cli',
         f'-j{os.cpu_count() or 4}'],
        cwd=LLAMA_DIR, check=True,
    )
    elapsed = time.time() - t0
    print(f'✓ llama.cpp built in {elapsed/60:.1f} min')

assert _q_bin.is_file(),   f'llama-quantize not found at {_q_bin}'
assert _cli_bin.is_file(), f'llama-cli not found at {_cli_bin}'
print(f'  llama-quantize : {_q_bin}')
print(f'  llama-cli      : {_cli_bin}')
print('\n✓ Cell 1 complete — proceed to Cell 2')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — HuggingFace login + download Gemma 4 12B
#
# You must accept the model terms at:
#   https://huggingface.co/google/gemma-4-12b-it
# before this cell will work.
#
# Time: 5–15 min depending on Colab bandwidth (~24 GB download)
# ══════════════════════════════════════════════════════════════════════════════
from huggingface_hub import login, snapshot_download
import os
from pathlib import Path

# ── Set your HF token here or paste when prompted ─────────────────────────────
# Option A: paste token (interactive)
# login()

# Option B: set HF_TOKEN secret in Colab Secrets (recommended)
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✓ HuggingFace login via HF_TOKEN')
else:
    print('⚠  HF_TOKEN not set — prompting interactively')
    login()

# ── Model config ──────────────────────────────────────────────────────────────
# Model card: https://huggingface.co/google/gemma-4-12b-it
# Verify the exact repo name matches what you accepted terms for.
MODEL_ID  = os.environ.get('SRD_MODEL_ID', 'google/gemma-4-12b-it')
MODEL_DIR = Path('/content/gemma4_12b')

print(f'\nModel  : {MODEL_ID}')
print(f'Dest   : {MODEL_DIR}')
print('Downloading weights (safetensors, ~24 GB) ...')

t0 = __import__('time').time()
if not MODEL_DIR.is_dir():
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=str(MODEL_DIR),
        local_dir_use_symlinks=False,
        token=HF_TOKEN or None,
    )
elapsed = __import__('time').time() - t0
size_gb = sum(f.stat().st_size for f in MODEL_DIR.rglob('*') if f.is_file()) / 1024**3
print(f'\n✓ Downloaded in {elapsed/60:.1f} min — {size_gb:.1f} GB')

# ── Print architecture info ───────────────────────────────────────────────────
import json
config_path = MODEL_DIR / 'config.json'
if config_path.is_file():
    cfg = json.loads(config_path.read_text())
    hidden = cfg.get('hidden_size', '?')
    layers = cfg.get('num_hidden_layers', '?')
    vocab  = cfg.get('vocab_size', '?')
    heads  = cfg.get('num_attention_heads', '?')
    ctx    = cfg.get('max_position_embeddings', '?')
    print(f'  Architecture : {cfg.get("model_type", "?")} ({cfg.get("architectures", ["?"])[0]})')
    print(f'  Hidden size  : {hidden}')
    print(f'  Layers       : {layers}')
    print(f'  Vocab size   : {vocab:,}' if isinstance(vocab, int) else f'  Vocab size   : {vocab}')
    print(f'  Attn heads   : {heads}')
    print(f'  Max context  : {ctx:,}' if isinstance(ctx, int) else f'  Max context  : {ctx}')
    # Compute embedding footprint
    if isinstance(vocab, int) and isinstance(hidden, int):
        emb_mb = vocab * hidden * 2 / 1024**2
        print(f'  Embedding F16: {emb_mb:.0f} MB  ({vocab:,} × {hidden} × 2 bytes)')
print('\n✓ Cell 2 complete — proceed to Cell 3')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — SRD-4 pack → gemma4_12b_srd4.axm  (~20–40 min on A100)
#
# SRD-4 = W4 base quantization, ~4.5 bpw.
# Signs every weight shard with HMAC-SHA256 — the fingerprint in the .axm
# is a public commitment to exactly these weights.
# ══════════════════════════════════════════════════════════════════════════════
import json, os, subprocess, sys, time
from pathlib import Path

REPO     = Path('/content/axiom')
OUT_DIR  = Path('/content')
AXM_PATH = OUT_DIR / 'gemma4_12b_srd4.axm'
RESULTS  = REPO / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

MODEL_DIR = Path('/content/gemma4_12b')
assert MODEL_DIR.is_dir(), 'Model not found — run Cell 2 first'
assert os.environ.get('AXIOM_MASTER_KEY'), 'AXIOM_MASTER_KEY not set — re-run Cell 1'

import torch
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    vram_gb = p.total_memory / 1024**3
    try:
        import psutil
        ram_avail = psutil.virtual_memory().available / 1024**3
    except Exception:
        ram_avail = 0
    print(f'GPU : {p.name}  {vram_gb:.1f} GB VRAM')
    print(f'RAM available: {ram_avail:.1f} GB')

print('\nPacking Gemma 4 12B → SRD-4 .axm ...')
print(f'  Input  : {MODEL_DIR}  (local path)')
print(f'  Output : {AXM_PATH}')
print(f'  Mode   : SRD-4 (W4 base, top_k_pct=0, ~4.5 bpw)')
print()

t0 = time.time()
subprocess.run(
    [sys.executable, 'axm_cli.py', 'pack',
     '--model',      str(MODEL_DIR),
     '--srd4',
     '--real-pack',
     '--output',     str(AXM_PATH),
     '--stats-json', str(RESULTS / 'gemma4_pack.json')],
    cwd=REPO, check=True,
)
elapsed = time.time() - t0

size_gb = AXM_PATH.stat().st_size / 1024**3
try:
    stats = json.loads((RESULTS / 'gemma4_pack.json').read_text())
except Exception:
    stats = {}

print(f'\n✓ Packed in {elapsed/60:.1f} min')
print(f'  .axm size   : {size_gb:.2f} GB')
print(f'  bpw         : {stats.get("quant", {}).get("bpw", "N/A")}')
print(f'  fingerprint : {stats.get("fingerprint", "N/A")}')
print('\n✓ Cell 3 complete — proceed to Cell 4')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Verify .axm proof ledger  (~10 s)
#
# Checks every HMAC-SHA256 proof in the container.
# If any weight shard was tampered with, this will fail.
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, sys
from pathlib import Path

REPO     = Path('/content/axiom')
AXM_PATH = Path('/content/gemma4_12b_srd4.axm')
assert AXM_PATH.is_file(), 'gemma4_12b_srd4.axm not found — run Cell 3 first'

print('Verifying gemma4_12b_srd4.axm ...')
result = subprocess.run(
    [sys.executable, 'axm_cli.py', 'verify', str(AXM_PATH)],
    cwd=REPO, capture_output=True, text=True,
)
try:
    output = json.loads(result.stdout)
except Exception:
    output = {'verified': False, 'error': result.stdout + result.stderr}

print(json.dumps(output, indent=2))

if not output.get('verified'):
    raise RuntimeError('\n✗ Verification FAILED — do not extract. Container may be tampered.')

FINGERPRINT = output.get('fingerprint', '')
print(f'\n✓ Verified  ({output.get("proofs_checked", "?")} proofs)')
print(f'  fingerprint : {FINGERPRINT}')
print('\n✓ Cell 4 complete — proceed to Cell 5')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Extract .axm → FP16 → GGUF Q4_K_M  (~15–25 min on A100)
#
# Pipeline:
#   .axm weights (SRD-4 W4)  →  reconstruct FP16  →  convert_hf_to_gguf.py
#   → model_f16.gguf  →  llama-quantize Q4_K_M  →  gemma4_12b_q4km.gguf
# ══════════════════════════════════════════════════════════════════════════════
import subprocess, sys, time
from pathlib import Path

REPO      = Path('/content/axiom')
AXM_PATH  = Path('/content/gemma4_12b_srd4.axm')
GGUF_PATH = Path('/content/gemma4_12b_q4km.gguf')
LLAMA_DIR = Path('/content/llama.cpp')

assert AXM_PATH.is_file(), '.axm not found — run Cell 3 first'
assert (LLAMA_DIR / 'build/bin/llama-quantize').is_file(), \
    'llama-quantize not found — re-run Cell 1'

print('Extracting .axm → GGUF Q4_K_M ...')
print(f'  Input  : {AXM_PATH}')
print(f'  Output : {GGUF_PATH}')
print(f'  Quant  : Q4_K_M')
print()

t0 = time.time()
subprocess.run(
    [sys.executable, '-m', 'research.quant.axm_to_gguf',
     '--container', str(AXM_PATH),
     '--gguf-out',  str(GGUF_PATH),
     '--llamacpp',  str(LLAMA_DIR),
     '--quant',     'Q4_K_M',
     '--device',    'cuda'],
    cwd=REPO, check=True,
)
elapsed = time.time() - t0
size_gb = GGUF_PATH.stat().st_size / 1024**3
print(f'\n✓ Extracted in {elapsed/60:.1f} min')
print(f'  GGUF size : {size_gb:.2f} GB  ({GGUF_PATH.name})')
print('\n✓ Cell 5 complete — proceed to Cell 6')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — Axiom MET metadata sidecar
#
# Reads model config to compute real layer/vocab/hidden numbers, then writes
# gemma4_12b_q4km.axiom_meta.json with:
#   - embedding slot size (F16, always pinned)
#   - transformer chunk boundaries (early / factual / reasoning / governance)
#   - hydration policy per intent class
#   - .axm fingerprint embedded
# ══════════════════════════════════════════════════════════════════════════════
import json, math, subprocess, sys
from pathlib import Path

REPO      = Path('/content/axiom')
GGUF_PATH = Path('/content/gemma4_12b_q4km.gguf')
MODEL_DIR = Path('/content/gemma4_12b')

assert GGUF_PATH.is_file(), 'GGUF not found — run Cell 5 first'

# ── Read actual model architecture ───────────────────────────────────────────
cfg_path = MODEL_DIR / 'config.json'
if cfg_path.is_file():
    cfg    = json.loads(cfg_path.read_text())
    HIDDEN = cfg.get('hidden_size', 4096)
    LAYERS = cfg.get('num_hidden_layers', 28)
    VOCAB  = cfg.get('vocab_size', 262144)
else:
    print('  [warn] config.json not found — using Gemma 4 12B defaults')
    HIDDEN, LAYERS, VOCAB = 4096, 28, 262144

# ── Compute embedding slot size ───────────────────────────────────────────────
# Gemma 4 uses a 256K token vocabulary. The embedding (tok_embeddings + lm_head,
# weight-tied) stays at F16 — it's the EventToken slot that is always pinned.
EMBEDDING_MB = round(VOCAB * HIDDEN * 2 / 1024**2, 1)   # F16 bytes

# ── Derive transformer chunk boundaries ───────────────────────────────────────
# Split LAYERS into 4 slots with proportions matching SmolLM2-135M:
#   early 20% · factual 20% · reasoning 37% · governance 23%
# These proportions are architecture-agnostic.
def _split(n, fracs):
    """Split n layers into len(fracs) non-overlapping ranges."""
    cuts, lo = [], 0
    for f in fracs:
        hi = lo + max(1, round(n * f))
        hi = min(hi, n)
        cuts.append((lo, hi - 1))
        lo = hi
    # Absorb rounding residual into last slot
    cuts[-1] = (cuts[-1][0], n - 1)
    return cuts

FRACS = [0.20, 0.20, 0.37, 0.23]
RANGES = _split(LAYERS, FRACS)
SLOT_NAMES = ['early', 'factual', 'reasoning', 'governance']
SLOT_RANGES = {name: rng for name, rng in zip(SLOT_NAMES, RANGES)}

# Estimate MB per chunk (Q4_K_M average ~4.5 bits/weight)
total_transformer_params = 12e9 - VOCAB * HIDDEN  # approx non-embedding params
for slot, (lo, hi) in SLOT_RANGES.items():
    n_layer_layers = hi - lo + 1
    frac = n_layer_layers / LAYERS
    chunk_mb = round(total_transformer_params * frac * 4.5 / 8 / 1024**2)
    SLOT_RANGES[slot] = {'layers': f'{lo}-{hi}', 'first_layer': lo, 'last_layer': hi,
                         'mb': chunk_mb, 'precision': 'Q4_K_M'}

HYDRATION_POLICY = {
    'INFORM':    ['early'],
    'CLARIFY':   ['early', 'governance'],
    'REFUSE':    ['early', 'governance'],
    'UNCERTAIN': ['early', 'governance'],
    'HARM':      ['early', 'factual', 'reasoning', 'governance'],
    'DECEIVE':   ['early', 'factual', 'reasoning', 'governance'],
}

# ── Build chunk map ───────────────────────────────────────────────────────────
chunk_map = {}
for slot, info in SLOT_RANGES.items():
    for idx in range(info['first_layer'], info['last_layer'] + 1):
        chunk_map[str(idx)] = slot

# ── Try to read fingerprint from Cell 4 output ───────────────────────────────
try:
    FINGERPRINT = FINGERPRINT  # set in cell 4
except NameError:
    FINGERPRINT = ''

# ── Compute intent RAM budgets ────────────────────────────────────────────────
slot_mb = {s: SLOT_RANGES[s]['mb'] for s in SLOT_NAMES}
import time
intent_ram = {}
STORAGE_MBS = 1500   # UFS 3.1 (Galaxy Fold 4 / Pixel)
for intent, chunks in HYDRATION_POLICY.items():
    xfmr_mb  = sum(slot_mb[c] for c in chunks)
    total_mb = EMBEDDING_MB + xfmr_mb
    load_ms  = round(xfmr_mb / STORAGE_MBS * 1000, 1)
    intent_ram[intent] = {
        'chunks': chunks, 'transformer_mb': xfmr_mb,
        'total_mb': total_mb, 'ufs_load_ms': load_ms,
    }

meta = {
    'axiom_version': '1.4',
    'generated_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'model_id': 'google/gemma-4-12b-it',
    'gguf_path': str(GGUF_PATH),
    'gguf_mb': round(GGUF_PATH.stat().st_size / 1024**2),
    'fingerprint': FINGERPRINT,
    'architecture': {'hidden_size': HIDDEN, 'num_layers': LAYERS, 'vocab_size': VOCAB},
    'embedding_slot': {
        'chunk': 'embedding', 'mb': EMBEDDING_MB, 'precision': 'F16',
        'always_pinned': True,
        'note': f'{VOCAB:,}-token vocabulary — embedding is {EMBEDDING_MB:.0f} MB F16',
    },
    'transformer_chunks': SLOT_RANGES,
    'chunk_map': chunk_map,
    'hydration_policy': HYDRATION_POLICY,
    'intent_ram_budget': intent_ram,
    'storage_speed_mbs': STORAGE_MBS,
    'between_met_floor_mb': EMBEDDING_MB,
    'peak_harm_mb': EMBEDDING_MB + sum(slot_mb.values()),
}

sidecar_path = GGUF_PATH.with_suffix('.axiom_meta.json')
sidecar_path.write_text(json.dumps(meta, indent=2) + '\n')
print(f'✓ Sidecar written → {sidecar_path}')

# ── Display summary ───────────────────────────────────────────────────────────
print()
print('MET SLOT SUMMARY  (Gemma 4 12B)')
print(f'  Architecture  : {LAYERS} layers · hidden {HIDDEN} · vocab {VOCAB:,}')
print()
print(f'  {"Slot":<14}  {"Layers":<10}  {"MB":>6}  {"Prec":>8}  Always?')
print('  ' + '─' * 52)
print(f'  {"embedding":<14}  {"—embed—":<10}  {EMBEDDING_MB:>6.0f}  {"F16":>8}  ✓ pinned')
for slot, info in SLOT_RANGES.items():
    print(f'  {slot:<14}  {info["layers"]:<10}  {info["mb"]:>6}  {"Q4_K_M":>8}  on demand')

print()
print('HYDRATION POLICY  (intent → transformer chunks)')
print(f'  {"Intent":<10}  {"Chunks":<36}  {"Total MB":>9}  {"UFS ms":>7}')
print('  ' + '─' * 68)
for intent, info in intent_ram.items():
    print(f'  {intent:<10}  {"+".join(info["chunks"]):<36}  {info["total_mb"]:>9}  {info["ufs_load_ms"]:>5.1f}ms')

print()
if FINGERPRINT:
    print(f'  Fingerprint : {FINGERPRINT}')
print()
print('PHONE DEPLOY  (Galaxy Fold 4 / Pixel with PocketPal):')
print(f'  adb push {GGUF_PATH.name} /storage/emulated/0/models/')
print(f'  adb push {sidecar_path.name} /storage/emulated/0/models/')
print('\n✓ Cell 6 complete — proceed to Cell 7')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — Smoke test + memory dashboard
#
# Runs a single 20-token generation through llama-cli to confirm the GGUF
# loads and generates without error, then prints the full MET memory budget.
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, time
from pathlib import Path

GGUF_PATH = Path('/content/gemma4_12b_q4km.gguf')
LLAMA_CLI = Path('/content/llama.cpp/build/bin/llama-cli')
assert GGUF_PATH.is_file(), 'GGUF not found — run Cell 5 first'
assert LLAMA_CLI.is_file(), 'llama-cli not found — re-run Cell 1'

# ── Smoke test: 20-token generation ──────────────────────────────────────────
PROMPT = '<start_of_turn>user\nSay AXIOM in one word only.<end_of_turn>\n<start_of_turn>model\n'
print('Smoke test (20 tokens) ...')
t0 = time.time()
result = subprocess.run(
    [str(LLAMA_CLI),
     '-m',  str(GGUF_PATH),
     '-p',  PROMPT,
     '-n',  '20',
     '--temp', '0.0',
     '-ngl', '99',    # offload all layers to GPU
     '--log-disable'],
    capture_output=True, text=True, timeout=120,
)
elapsed = time.time() - t0
if result.returncode != 0:
    print(f'✗ smoke test failed:\n{result.stderr[-500:]}')
else:
    output_text = result.stdout.strip()
    print(f'  Output   : {repr(output_text[-120:])}')
    print(f'  Time     : {elapsed:.1f}s')
    print('  ✓ GGUF loads and generates correctly')

# ── Read sidecar for memory dashboard ────────────────────────────────────────
sidecar_path = GGUF_PATH.with_suffix('.axiom_meta.json')
if sidecar_path.is_file():
    meta = json.loads(sidecar_path.read_text())
else:
    print('  [warn] sidecar not found — run Cell 6 to generate MET metadata')
    meta = {}

if meta:
    EMB_MB    = meta['embedding_slot']['mb']
    PEAK_MB   = meta['peak_harm_mb']
    GGUF_MB   = meta['gguf_mb']
    ir        = meta.get('intent_ram_budget', {})
    W         = 32

    def bar(mb, total=None):
        total = total or PEAK_MB
        n = round(min(mb / total, 1.0) * W)
        return '█' * n + '░' * (W - n)

    print()
    print('═' * 72)
    print('  AXIOM MET HYDRATION MEMORY DASHBOARD  —  Gemma 4 12B Q4_K_M')
    print('─' * 72)
    arch = meta.get('architecture', {})
    print(f'  Model    : google/gemma-4-12b-it  {arch.get("num_layers","?")} layers '
          f'{arch.get("hidden_size","?")} hidden  {arch.get("vocab_size",0):,}-token vocab')
    print(f'  GGUF     : {GGUF_MB:,} MB  ({GGUF_MB/1024:.1f} GB)')
    print(f'  Fingerprint: {meta.get("fingerprint") or "(not set)"}')
    print()
    print(f'  {"Static (full GGUF in RAM)":<36}  {GGUF_MB:>6} MB  {bar(GGUF_MB)}')
    print(f'  {"Embedding floor (between METs)":<36}  {EMB_MB:>6.0f} MB  {bar(EMB_MB)}')
    print(f'  {"Peak (HARM full hydration)":<36}  {PEAK_MB:>6.0f} MB  {bar(PEAK_MB)}')
    print()
    print(f'  {"Intent":<10}  {"RAM MB":>7}  {"vs static":>10}  Bar')
    print('  ' + '─' * 62)
    for intent, info in ir.items():
        tmb    = info['total_mb']
        saving = (1 - tmb / GGUF_MB) * 100
        marker = '◄ HARM' if intent in ('HARM','DECEIVE') else ''
        print(f'  {intent:<10}  {tmb:>7.0f}  {saving:>9.0f}%  {bar(tmb)}  {marker}')
    print()
    avg_70_20_9_1 = (0.70 * ir.get('INFORM',{}).get('total_mb',0)
                   + 0.20 * ir.get('CLARIFY',{}).get('total_mb',0)
                   + 0.09 * ir.get('REFUSE',{}).get('total_mb',0)
                   + 0.01 * ir.get('HARM',{}).get('total_mb',0))
    saving_avg = (1 - avg_70_20_9_1 / GGUF_MB) * 100
    print(f'  Avg RAM (70/20/9/1 workload) : {avg_70_20_9_1:.0f} MB  —  {saving_avg:.0f}% below static GGUF')
    print(f'  Between-MET floor            : {EMB_MB:.0f} MB  (embedding in EventToken slot)')
    print()
    print('  TRANSFORMER CHUNK BOUNDARIES  (for QRF pre-hydration)')
    print(f'  {"Slot":<14}  {"Layers":<10}  {"MB":>6}  {"Precision"}')
    print('  ' + '─' * 44)
    print(f'  {"embedding":<14}  {"—embed—":<10}  {EMB_MB:>6.0f}  F16  ← always pinned')
    for slot, info in meta.get('transformer_chunks', {}).items():
        print(f'  {slot:<14}  {info["layers"]:<10}  {info["mb"]:>6}  Q4_K_M')
    print('═' * 72)

print('\n✓ Pipeline complete.')
print(f'  GGUF   : {GGUF_PATH}')
sidecar = GGUF_PATH.with_suffix('.axiom_meta.json')
if sidecar.is_file():
    print(f'  Sidecar: {sidecar}')
print()
print('Download files:')
print('  from google.colab import files')
print(f'  files.download("{GGUF_PATH}")')
print(f'  files.download("{sidecar}")')